# 2.10 Dual basis functions
We use dual basis functions to define interpolation operators, transfer operators between different finite element spaces, and auxiliary space preconditioners.

### Canonical interpolation
The canonical finite element interpolation operator is defined by specifying the degrees of freedom. For low order methods these are typically nodal values, while for high order methods these are most often moments. For example, the interpolation of a function $u$ onto the $p^{th}$ order triangle is given by: find $u_{hp} \in V_{hp}$ such that

\begin{eqnarray*}
u_{hp} (V) & = & u(V) \quad \forall \text{ vertices } V \\
\int_E u_{hp} q & = & \int_E u q \quad \forall q \in P^{p-2}(E) \; \forall \text{ edges } E \\
\int_T u_{hp} q & = & \int_T u q \quad \forall q \in P^{p-3}(T) \; \forall \text{ triangles } T
\end{eqnarray*}

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw

import matplotlib.pylab as plt
mesh = Mesh(unit_square.GenerateMesh(maxh=2))

The NGSolve `Set` function does local projection, and simple averaging. In particular, this does not respect point values in mesh vertices.

In [ ]:
fes = H1(mesh, order=3, low_order_space=False)

func = x*x*x*x
gfu = GridFunction(fes)
gfu.Set(func)
Draw (gfu)
print (gfu.vec)

Most NGSolve finite element spaces provide a `dual` operator, which delivers the moments (i.e. the dual space basis functions) instead of function values. The integrals over faces, edges and also vertices are defined by co-dimension 1 (=BND), co-dimension 2 (=BBND) or co-dimension 3 (=BBBND) integrals over the volume elements.
We define a variational problem for canonical interpolation:

In [ ]:
u,v = fes.TnT()
vdual = v.Operator("dual")

a = BilinearForm(u*vdual*dx + u*vdual*dx(element_vb=BND) + \
                 u*vdual*dx(element_vb=BBND)).Assemble()

f = LinearForm(func*vdual*dx + func*vdual*dx(element_vb=BND) + \
               func*vdual*dx(element_vb=BBND)).Assemble()

# interpolation in vertices preserves values 0 and 1
gfu.vec.data = a.mat.Inverse() * f.vec
print (gfu.vec)
Draw (gfu);

The vertex degrees of freedom vanish for edge and element basis functions, and the edge degrees of freedom vanish for element basis functions, but not vice versa. Thus, the obtained matrix A is block-triangular:

In [ ]:
import scipy.sparse as sp
A = sp.csr_matrix(a.mat.CSR())
plt.rcParams['figure.figsize'] = (4,4)
plt.spy(A)
plt.show()

We can use block Gauss-Seidel smoothing for solving with that block-triangular matrix by blocking the dofs of the individual vertices, edges and elements. Since the NGSolve Gauss-Seidel smoother reorders the smoothing blocks for parallelization, we have to take care to first compute vertex values, then edge values, and finally element values by running three different Gauss-Seidel sweeps.

In [ ]:
vblocks = [fes.GetDofNrs(vertex) for vertex in mesh.vertices]
eblocks = [fes.GetDofNrs(edge) for edge in mesh.edges]
fblocks = [fes.GetDofNrs(face) for face in mesh.faces]

print (vblocks)
print (eblocks)
print (fblocks)

vinv = a.mat.CreateBlockSmoother(vblocks)
einv = a.mat.CreateBlockSmoother(eblocks)
finv = a.mat.CreateBlockSmoother(fblocks)

vinv.Smooth(gfu.vec, f.vec)
einv.Smooth(gfu.vec, f.vec)
finv.Smooth(gfu.vec, f.vec)
print (gfu.vec)

This canonical interpolation is available as the `Interpolate` method of the `GridFunction`:

In [ ]:
gfu.Interpolate(func)
print (gfu.vec)

## Embedding Finite Element Spaces
This interpolation can be used to transform functions from one finite element space $V_{src}$ to another one $V_{dst}$. We use the dual space of the destination space:

$$
\int_{node} u_{dst} v_{dual} = \int_{node} u_{src} v_{dual} \qquad
\forall \, v_{dual} \; \forall \, \text{nodes}
$$

The left hand side leads to a non-symmetric square matrix, the right hand side to a rectangular matrix.

As an example we implement the transformation from a vector-valued $H^1$ space into $H(\operatorname{div})$:

In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.2))

fesh1 = VectorH1(mesh, order=2)
feshdiv = HDiv(mesh, order=2)

gfuh1 = GridFunction(fesh1)
gfuh1.Set ( CF((x*x,y*y)) )

gfuhdiv = GridFunction(feshdiv, name="uhdiv")

Build the matrices, and use a direct solver:

In [ ]:
u,v = feshdiv.TnT()
vdual = v.Operator("dual")
uh1 = fesh1.TrialFunction()

dS = dx(element_boundary=True)
ahdiv = BilinearForm(u*vdual*dx + u*vdual*dS).Assemble()

amixed = BilinearForm(uh1*vdual*dx + uh1*vdual*dS).Assemble()

# build transformation operator:
transform = ahdiv.mat.Inverse() @ amixed.mat
gfuhdiv.vec.data = transform * gfuh1.vec

Draw (gfuh1)
Draw (gfuhdiv);

We implement a linear operator performing the fast conversion by Gauss-Seidel smoothing:

In [ ]:
class MyBlockInverse(BaseMatrix):
    def __init__ (self, mat, eblocks, fblocks):
        super().__init__()
        self.mat = mat
        self.einv = mat.CreateBlockSmoother(eblocks)
        self.finv = mat.CreateBlockSmoother(fblocks)
        self.res = self.mat.CreateColVector()

    def CreateRowVector(self):
        return self.mat.CreateColVector()
    def CreateColVector(self):
        return self.mat.CreateRowVector()

    def Mult(self, x, y):
        # y[:] = 0
        # self.einv.Smooth(y,x)    #   y = y +  A_E^-1  (x - A y)
        # self.finv.Smooth(y,x)    #   y = y +  A_F^-1  (x - A y)

        # the same, but we see how to transpose that
        y.data = self.einv * x
        self.res.data = x - self.mat * y
        y.data += self.finv * self.res

    def MultTrans(self, x, y):
        y.data = self.finv.T * x
        self.res.data = x - self.mat.T * y
        y.data += self.einv.T * self.res


eblocks = [feshdiv.GetDofNrs(edge) for edge in mesh.edges]
fblocks = [feshdiv.GetDofNrs(face) for face in mesh.faces]

transform = MyBlockInverse(ahdiv.mat, eblocks, fblocks) @ amixed.mat
gfuhdiv.vec.data = transform * gfuh1.vec

Such transformation operators between finite element spaces are provided by `ConvertOperator`:

In [ ]:
convert = ConvertOperator(fesh1, feshdiv)
hv = gfuhdiv.vec.CreateVector()
hv.data = convert * gfuh1.vec
print ("difference:", Norm(hv - gfuhdiv.vec))

## Auxiliary Space Preconditioning
Nepomnyaschikh 91, Hiptmair-Xu 07, ....

Assume we have a complicated problem with some complicated discretization, and we have good preconditioners for a nodal finite element discretization of the Laplace operator. By auxiliary space preconditioning we can reuse the simple preconditioners for the complicated problems. It is simple, and works well in many cases.

As a simple example, we precondition a DG discretization by an $H^1$-conforming method.

In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))

The DG discretization:

In [ ]:
order=4
fesDG = L2(mesh, order=order, dgjumps=True)
u,v = fesDG.TnT()
jump_u = u-u.Other()
jump_v = v-v.Other()
n = specialcf.normal(mesh.dim)
mean_dudn = 0.5*n * (grad(u)+grad(u.Other()))
mean_dvdn = 0.5*n * (grad(v)+grad(v.Other()))
alpha = 4
h = specialcf.mesh_size
aDG = BilinearForm(fesDG)
aDG += grad(u)*grad(v) * dx
aDG += alpha*order**2/h*jump_u*jump_v * dx(skeleton=True)
aDG += alpha*order**2/h*u*v * ds(skeleton=True)
aDG += (-mean_dudn*jump_v -mean_dvdn*jump_u) * dx(skeleton=True)
aDG += (-n*grad(u)*v-n*grad(v)*u)* ds(skeleton=True)
aDG.Assemble()

fDG = LinearForm(1*v*dx).Assemble()
gfuDG = GridFunction(fesDG)

The auxiliary $H^1$ discretization:

In [ ]:
fesH1 = H1(mesh, order=2, dirichlet=".*")
u,v = fesH1.TnT()
aH1 = BilinearForm(grad(u)*grad(v)*dx)
preH1 = preconditioners.BDDC(aH1)
aH1.Assemble();

The auxiliary space preconditioner is the sum of the $H^1$ preconditioner, transferred to the DG space, and a simple smoother for the DG matrix:

In [ ]:
transform = ConvertOperator(fesH1, fesDG, geom_free=True)
pre = transform @ preH1 @ transform.T + aDG.mat.CreateSmoother()

inv = solvers.CGSolver(aDG.mat, pre, maxiter=200)
gfuDG.vec.data = inv * fDG.vec
print ("CG iterations:", inv.iterations)
Draw (gfuDG);

For comparison, the iteration with the smoother alone:

In [ ]:
inv = solvers.CGSolver(aDG.mat, aDG.mat.CreateSmoother(), maxiter=2000)
hv = (inv * fDG.vec).Evaluate()
print ("CG iterations:", inv.iterations)